# 3.1 Demographic, SBR, and Feature Analysis (with PATNO Matching)

This notebook investigates the relationship between:
- **Patient Demographics**: Age and Sex
- **SBR Pathology Targets**: The three SBR Principal Component scores (SBR_PC1, SBR_PC2, SBR_PC3)
- **Imaging Features**: The 256-dimensional latent vectors extracted from the weighted autoencoder

**Key Improvement**: This version properly merges datasets using PATNO extracted from file paths.

## Objectives
1. Evaluate how well imaging features predict SBR pathology targets
2. Assess the added value of demographic covariates (Age, Sex) in prediction
3. Compare model performance across the three SBR principal components

## Section 1: Setup and Data Preparation

### 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

### 1.2 Define Target and Feature Sets

In [ ]:
# Target variables: SBR Principal Components
TARGETS = ['SBR_PC1', 'SBR_PC2', 'SBR_PC3']

# Demographic covariates
DEMO_COLS = ['AGE_AT_VISIT', 'SEX']

# Image feature columns (256-dimensional latent vectors)
FEATURE_COLS = [f'latent_{i}' for i in range(256)]

print(f"Target variables: {TARGETS}")
print(f"Demographic columns: {DEMO_COLS}")
print(f"Number of image features: {len(FEATURE_COLS)}")

### 1.3 Load Data

In [ ]:
# Load merged clinical/demographic data with SBR PCA scores
df_merged = pd.read_csv('output/merged_data.csv')
print(f"Merged clinical data shape: {df_merged.shape}")
print(f"Available columns: {df_merged.columns.tolist()[:15]}...")  # Show first 15 columns

# Check if SBR_PC columns exist
sbr_pc_cols = [col for col in df_merged.columns if col.startswith('SBR_PC')]
print(f"\nSBR PC columns found: {sbr_pc_cols}")

# Load latent vectors WITH PATNO (validation set)
latent_file = 'output/Experiments/LatentVectorAnalysis/validation_latent_vectors_with_patno.csv'
if Path(latent_file).exists():
    df_latent = pd.read_csv(latent_file)
    print(f"\nLatent vectors with PATNO loaded: {df_latent.shape}")
    print(f"Columns: {df_latent.columns.tolist()[-5:]}...")  # Show last 5 columns
else:
    print(f"\n⚠️ WARNING: {latent_file} not found!")
    print("Please run: python extract_latent_with_patno.py")
    raise FileNotFoundError(f"Required file not found: {latent_file}")

### 1.4 Merge Latent Vectors with Clinical Data using PATNO

In [ ]:
# Check PATNO availability
print("PATNO Statistics:")
print(f"  Clinical data - Unique PATNOs: {df_merged['PATNO'].nunique()}")
print(f"  Latent vectors - Unique PATNOs: {df_latent['PATNO'].nunique()}")
print(f"  Latent vectors - Missing PATNOs: {df_latent['PATNO'].isna().sum()}")

# Remove rows with missing PATNO from latent vectors
df_latent_clean = df_latent.dropna(subset=['PATNO']).copy()
print(f"\nLatent vectors after removing missing PATNOs: {len(df_latent_clean)}")

# Merge on PATNO
# Note: Clinical data may have multiple visits per patient, so we'll need to handle this
print("\nMerging datasets on PATNO...")
df_combined = pd.merge(
    df_latent_clean,
    df_merged,
    on='PATNO',
    how='inner'
)

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Unique patients in combined data: {df_combined['PATNO'].nunique()}")

# Check for multiple visits per patient
visits_per_patient = df_combined.groupby('PATNO').size()
print(f"\nVisits per patient statistics:")
print(visits_per_patient.describe())
print(f"Patients with multiple visits: {(visits_per_patient > 1).sum()}")

### 1.5 Handle Multiple Visits per Patient

Since each imaging scan corresponds to one latent vector, but patients may have multiple clinical visits, we need to select the most appropriate visit for each scan.

In [ ]:
# Strategy: For each patient, select the visit with non-null SBR data that's closest to baseline
# Priority: SC (Screening) > BL (Baseline) > other visits

def select_best_visit(group):
    """Select the best visit for each patient based on data availability and visit type"""
    # Filter to rows with all required data
    valid_rows = group.dropna(subset=TARGETS + DEMO_COLS)
    
    if len(valid_rows) == 0:
        return None
    
    # Priority order for EVENT_ID
    if 'EVENT_ID' in valid_rows.columns:
        priority = {'SC': 0, 'BL': 1, 'V01': 2, 'V02': 3}
        valid_rows['priority'] = valid_rows['EVENT_ID'].map(priority).fillna(99)
        best_row = valid_rows.sort_values('priority').iloc[0]
    else:
        best_row = valid_rows.iloc[0]
    
    return best_row

print("Selecting best visit for each patient...")
df_final = df_combined.groupby('PATNO', group_keys=False).apply(select_best_visit)
df_final = df_final.reset_index(drop=True)

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Unique patients: {df_final['PATNO'].nunique()}")
print(f"Samples with complete data: {len(df_final)}")

### 1.6 Extract Features and Targets

In [ ]:
# Extract image features (X)
X_features = df_final[FEATURE_COLS].values
print(f"Image features (X) shape: {X_features.shape}")

# Extract target variables (Y)
Y_targets = df_final[TARGETS].values
print(f"Target variables (Y) shape: {Y_targets.shape}")

# Extract demographics
demo_data = df_final[DEMO_COLS].copy()
print(f"Demographics shape: {demo_data.shape}")

# Summary statistics
print(f"\nTarget variable statistics:")
print(df_final[TARGETS].describe())

print(f"\nDemographic statistics:")
print(f"Age - Mean: {demo_data['AGE_AT_VISIT'].mean():.2f}, Std: {demo_data['AGE_AT_VISIT'].std():.2f}")
print(f"Sex distribution:\n{demo_data['SEX'].value_counts()}")

### 1.7 Covariate Processing

In [ ]:
# One-Hot Encode SEX column
# SEX: 0 = Female, 1 = Male
sex_values = demo_data['SEX'].values.reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False, drop='first')  # Drop first to avoid multicollinearity
sex_encoded = encoder.fit_transform(sex_values)
print(f"Sex encoded shape: {sex_encoded.shape}")
print(f"Sex categories: {encoder.categories_}")

# Standardize AGE column
age_values = demo_data['AGE_AT_VISIT'].values.reshape(-1, 1)
age_scaler = StandardScaler()
age_scaled = age_scaler.fit_transform(age_values)
print(f"\nAge scaled shape: {age_scaled.shape}")
print(f"Age mean: {age_scaler.mean_[0]:.2f}, std: {age_scaler.scale_[0]:.2f}")

# Combine demographic features into Z_demo
Z_demo = np.hstack([age_scaled, sex_encoded])
print(f"\nCombined demographic features (Z_demo) shape: {Z_demo.shape}")
print(f"Z_demo columns: ['AGE_scaled', 'SEX_Male']")

### 1.8 Standardize Image Features

In [ ]:
# Standardize image features
feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X_features)

print(f"Standardized image features shape: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean():.6f}, Std: {X_scaled.std():.6f}")

## Section 2: Model Training and Evaluation

### 2.1 Initialize Results Storage

In [ ]:
# Dictionary to store results
results = {
    'Target': [],
    'Model': [],
    'R2_Score': [],
    'MSE': [],
    'RMSE': []
}

# Ridge regression hyperparameter
ALPHA = 1.0  # Regularization strength
TEST_SIZE = 0.2
RANDOM_STATE = 42

print(f"Ridge alpha: {ALPHA}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")

### 2.2 Train and Evaluate Models for Each Target

In [ ]:
for idx, target_name in enumerate(TARGETS):
    print("=" * 80)
    print(f"TARGET: {target_name}")
    print("=" * 80)
    
    # Extract target variable
    y = Y_targets[:, idx]
    
    # -------------------------------------------------------------------------
    # Model Set A: Image Features Only
    # -------------------------------------------------------------------------
    print(f"\n[Model A] Image Features Only")
    
    # Split data
    X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
        X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    # Train Ridge regression
    model_A = Ridge(alpha=ALPHA, random_state=RANDOM_STATE)
    model_A.fit(X_train_A, y_train_A)
    
    # Predict and evaluate
    y_pred_A = model_A.predict(X_test_A)
    r2_A = r2_score(y_test_A, y_pred_A)
    mse_A = mean_squared_error(y_test_A, y_pred_A)
    rmse_A = np.sqrt(mse_A)
    
    print(f"  R² Score: {r2_A:.4f}")
    print(f"  MSE: {mse_A:.4f}")
    print(f"  RMSE: {rmse_A:.4f}")
    
    # Store results
    results['Target'].append(target_name)
    results['Model'].append('Image Features Only')
    results['R2_Score'].append(r2_A)
    results['MSE'].append(mse_A)
    results['RMSE'].append(rmse_A)
    
    # -------------------------------------------------------------------------
    # Model Set B: Image Features + Demographics
    # -------------------------------------------------------------------------
    print(f"\n[Model B] Image Features + Demographics")
    
    # Combine features
    X_combined = np.hstack([X_scaled, Z_demo])
    print(f"  Combined features shape: {X_combined.shape}")
    
    # Split data
    X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
        X_combined, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    # Train Ridge regression
    model_B = Ridge(alpha=ALPHA, random_state=RANDOM_STATE)
    model_B.fit(X_train_B, y_train_B)
    
    # Predict and evaluate
    y_pred_B = model_B.predict(X_test_B)
    r2_B = r2_score(y_test_B, y_pred_B)
    mse_B = mean_squared_error(y_test_B, y_pred_B)
    rmse_B = np.sqrt(mse_B)
    
    print(f"  R² Score: {r2_B:.4f}")
    print(f"  MSE: {mse_B:.4f}")
    print(f"  RMSE: {rmse_B:.4f}")
    
    # Store results
    results['Target'].append(target_name)
    results['Model'].append('Image Features + Demographics')
    results['R2_Score'].append(r2_B)
    results['MSE'].append(mse_B)
    results['RMSE'].append(rmse_B)
    
    # Calculate improvement
    r2_improvement = r2_B - r2_A
    mse_reduction = ((mse_A - mse_B) / mse_A) * 100 if mse_A != 0 else 0
    
    print(f"\n[Improvement]")
    print(f"  ΔR²: {r2_improvement:+.4f} ({r2_improvement/r2_A*100:+.2f}%)" if r2_A != 0 else f"  ΔR²: {r2_improvement:+.4f}")
    print(f"  MSE Reduction: {mse_reduction:.2f}%")
    print()

print("\n" + "=" * 80)
print("MODEL TRAINING COMPLETE")
print("=" * 80)

## Section 3: Result Summary and Interpretation

### 3.1 Results Summary Table

In [ ]:
# Create results DataFrame
df_results = pd.DataFrame(results)

# Pivot for better visualization
df_pivot_r2 = df_results.pivot(index='Target', columns='Model', values='R2_Score')
df_pivot_mse = df_results.pivot(index='Target', columns='Model', values='MSE')
df_pivot_rmse = df_results.pivot(index='Target', columns='Model', values='RMSE')

print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)

print("\nR² Scores:")
print(df_pivot_r2.to_string())

print("\n\nMean Squared Error (MSE):")
print(df_pivot_mse.to_string())

print("\n\nRoot Mean Squared Error (RMSE):")
print(df_pivot_rmse.to_string())

### 3.2 Calculate Improvement Metrics

In [ ]:
# Calculate improvement for each target
improvement_data = []

for target in TARGETS:
    r2_img_only = df_pivot_r2.loc[target, 'Image Features Only']
    r2_combined = df_pivot_r2.loc[target, 'Image Features + Demographics']
    
    mse_img_only = df_pivot_mse.loc[target, 'Image Features Only']
    mse_combined = df_pivot_mse.loc[target, 'Image Features + Demographics']
    
    r2_delta = r2_combined - r2_img_only
    r2_pct_change = (r2_delta / r2_img_only * 100) if r2_img_only != 0 else 0
    mse_reduction = ((mse_img_only - mse_combined) / mse_img_only * 100) if mse_img_only != 0 else 0
    
    improvement_data.append({
        'Target': target,
        'R²_Image_Only': r2_img_only,
        'R²_Combined': r2_combined,
        'ΔR²': r2_delta,
        'R²_Change_%': r2_pct_change,
        'MSE_Reduction_%': mse_reduction
    })

df_improvement = pd.DataFrame(improvement_data)

print("\n" + "=" * 80)
print("IMPROVEMENT ANALYSIS")
print("=" * 80)
print(df_improvement.to_string(index=False))

### 3.3 Visualization: Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: R² Scores
df_pivot_r2.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], alpha=0.8, edgecolor='black')
axes[0].set_title('R² Score Comparison Across Targets', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Target Variable', fontsize=12)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].legend(title='Model', fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Plot 2: MSE
df_pivot_mse.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], alpha=0.8, edgecolor='black')
axes[1].set_title('Mean Squared Error Comparison Across Targets', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Target Variable', fontsize=12)
axes[1].set_ylabel('MSE', fontsize=12)
axes[1].legend(title='Model', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('output/demographic_sbr_feature_analysis_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to: output/demographic_sbr_feature_analysis_comparison.png")

### 3.4 Visualization: Improvement Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: R² Change
axes[0].bar(df_improvement['Target'], df_improvement['ΔR²'], 
            color='seagreen', alpha=0.8, edgecolor='black')
axes[0].set_title('R² Improvement from Adding Demographics', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Target Variable', fontsize=12)
axes[0].set_ylabel('ΔR² (Combined - Image Only)', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add value labels
for i, (target, delta) in enumerate(zip(df_improvement['Target'], df_improvement['ΔR²'])):
    axes[0].text(i, delta, f'{delta:+.4f}', ha='center', va='bottom' if delta > 0 else 'top', fontsize=10)

# Plot 2: MSE Reduction Percentage
axes[1].bar(df_improvement['Target'], df_improvement['MSE_Reduction_%'], 
            color='indianred', alpha=0.8, edgecolor='black')
axes[1].set_title('MSE Reduction from Adding Demographics', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Target Variable', fontsize=12)
axes[1].set_ylabel('MSE Reduction (%)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add value labels
for i, (target, reduction) in enumerate(zip(df_improvement['Target'], df_improvement['MSE_Reduction_%'])):
    axes[1].text(i, reduction, f'{reduction:.2f}%', ha='center', va='bottom' if reduction > 0 else 'top', fontsize=10)

plt.tight_layout()
plt.savefig('output/demographic_sbr_feature_analysis_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to: output/demographic_sbr_feature_analysis_improvement.png")

### 3.5 Interpretation and Key Findings

## Interpretation

### Key Questions Addressed:

#### 1. For which SBR_PC target did the image features show the strongest predictive power?

**Answer**: Examine the R² scores for **Model A (Image Features Only)** across all three targets:
- The target with the **highest R² score** indicates the strongest relationship between imaging features and that particular SBR principal component.
- Typically, **SBR_PC1** (representing overall dopamine transporter loss severity) shows the strongest predictive power, as it captures the most variance in the original SBR measurements and is most directly related to structural brain changes visible in imaging.

#### 2. How much did the addition of Age and Sex improve the prediction of each SBR_PC target?

**Answer**: Compare **Model A vs. Model B** for each target:
- **ΔR²** (R² improvement): Positive values indicate that demographics add predictive value.
- **MSE Reduction %**: Shows the percentage decrease in prediction error when demographics are included.

**Expected Patterns**:
- **SBR_PC1** (Overall Severity): Age is known to correlate with dopamine transporter loss, so we expect moderate improvement when Age is added.
- **SBR_PC2** (Asymmetry): This component may show less improvement from demographics, as asymmetry is more related to disease-specific patterns than age or sex.
- **SBR_PC3** (Regional Pattern): Improvement depends on whether regional loss patterns vary systematically with age or sex.

### Clinical Implications:

1. **Image Features as Primary Predictors**: If R² scores for Model A are already high (e.g., > 0.6), it suggests that the latent imaging features capture most of the pathology-relevant information.

2. **Value of Demographics**: If adding demographics provides substantial improvement (e.g., ΔR² > 0.05), it indicates that age and sex contribute independent information not fully captured by the imaging features alone.

3. **Target-Specific Insights**: Different SBR_PC targets may benefit differently from demographic covariates, suggesting that some aspects of pathology are more age/sex-dependent than others.

### Recommendations for Further Analysis:

- **Feature Importance**: Examine Ridge regression coefficients to identify which latent dimensions are most predictive.
- **Cross-Validation**: Implement k-fold cross-validation for more robust performance estimates.
- **Non-Linear Models**: Test whether non-linear models (e.g., Random Forest, Gradient Boosting) capture additional relationships.
- **Interaction Effects**: Investigate whether Age × Sex interactions improve prediction.

### 3.6 Save Results

In [ ]:
# Save results to CSV
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

df_results.to_csv(output_dir / 'demographic_sbr_feature_analysis_results.csv', index=False)
df_improvement.to_csv(output_dir / 'demographic_sbr_feature_analysis_improvement.csv', index=False)
df_final.to_csv(output_dir / 'merged_imaging_clinical_data.csv', index=False)

print("Results saved to:")
print(f"  - {output_dir / 'demographic_sbr_feature_analysis_results.csv'}")
print(f"  - {output_dir / 'demographic_sbr_feature_analysis_improvement.csv'}")
print(f"  - {output_dir / 'merged_imaging_clinical_data.csv'}")

## Summary

This notebook successfully:
1. ✅ Loaded latent vectors WITH PATNO information extracted from file paths
2. ✅ Merged imaging features with clinical data using proper PATNO matching
3. ✅ Handled multiple visits per patient by selecting the most appropriate visit
4. ✅ Processed demographic covariates (Age standardization, Sex one-hot encoding)
5. ✅ Trained Ridge regression models for each SBR_PC target with two feature sets:
   - Model A: Image Features Only
   - Model B: Image Features + Demographics
6. ✅ Evaluated models using R² and MSE metrics
7. ✅ Compared model performance and quantified the added value of demographics
8. ✅ Visualized results and provided clinical interpretation

**Key Improvement over 3.0**: This version properly merges datasets using PATNO, ensuring that imaging features are correctly matched with clinical/demographic data.

**Next Steps**:
- Investigate feature importance to identify key latent dimensions
- Explore non-linear models for potential performance gains
- Validate findings on independent test set
- Analyze prediction errors to identify systematic biases